# Ejercicio RESUELTO: Agente de Soporte IT con LangGraph

> **Solo para el profesor.** No compartir con alumnos hasta que hayan entregado.

---

Agente completo que clasifica tickets IT, busca la solución correcta con la herramienta adecuada y redacta una respuesta.

## Paso 0 — Instalación y configuración

In [ ]:
!pip install -q langchain langchain-openai langgraph

In [ ]:
import os
import logging

os.environ["OPENAI_API_KEY"] = "sk-..."  # tu clave de OpenAI
os.environ["LANGSMITH_TRACING"] = "false"
logging.getLogger("langsmith").setLevel(logging.CRITICAL)

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from typing import TypedDict, Annotated

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("Listo.")

## Paso 1 — Estado

In [ ]:
class EstadoIT(TypedDict):
    messages:  Annotated[list, add_messages]
    categoria: str   # "software", "red" o "hardware"
    solucion:  str   # la solución encontrada por la herramienta

## Paso 2 — Herramientas

In [ ]:
@tool
def buscar_solucion_software(problema: str) -> str:
    """Busca soluciones para problemas de software: aplicaciones, instalaciones, licencias, crashes."""
    soluciones = {
        "licencia": "1. Verifica que la licencia no haya expirado en el portal de IT.\n2. Desinstala y reinstala con el instalador del servidor interno.\n3. Si persiste, abre ticket en helpdesk@empresa.com.",
        "crash":    "1. Actualiza la aplicación a la última versión.\n2. Borra la caché en %AppData%.\n3. Reinstala con permisos de administrador.",
        "default":  "1. Reinicia la aplicación.\n2. Comprueba que el sistema cumple los requisitos mínimos.\n3. Contacta con soporte técnico si el error persiste.",
    }
    p = problema.lower()
    if "licencia" in p or "license" in p:
        return soluciones["licencia"]
    elif "crash" in p or "cierra" in p or "error" in p:
        return soluciones["crash"]
    return soluciones["default"]


@tool
def buscar_solucion_red(problema: str) -> str:
    """Busca soluciones para problemas de red: conexión, VPN, WiFi, DNS."""
    soluciones = {
        "wifi":    "1. Desconecta y reconecta al WiFi.\n2. Olvida la red y vuelve a conectarte.\n3. Reinicia el router si eres el único afectado.",
        "vpn":     "1. Cierra y vuelve a abrir el cliente VPN.\n2. Comprueba que tus credenciales no hayan expirado.\n3. Intenta conectarte desde otra red.",
        "default": "1. Ejecuta el diagnóstico de red de Windows (ipconfig /release && ipconfig /renew).\n2. Comprueba si otros compañeros tienen el mismo problema.\n3. Contacta con el departamento de redes.",
    }
    p = problema.lower()
    if "wifi" in p or "inalámbr" in p:
        return soluciones["wifi"]
    elif "vpn" in p:
        return soluciones["vpn"]
    return soluciones["default"]


@tool
def buscar_solucion_hardware(problema: str) -> str:
    """Busca soluciones para problemas de hardware: ordenador lento, ruidos, pantalla, periféricos."""
    soluciones = {
        "lento":   "1. Cierra las aplicaciones que no uses.\n2. Comprueba el uso de CPU y RAM en el Task Manager.\n3. Si el disco está al 100%, puede necesitar limpieza o sustitución.",
        "ruido":   "1. Limpia el ventilador con aire comprimido.\n2. Comprueba la temperatura con HWMonitor.\n3. Si el ruido persiste, el ventilador puede necesitar sustitución — abre ticket de hardware.",
        "default": "1. Reinicia el equipo.\n2. Comprueba las conexiones físicas.\n3. Si el problema persiste, abre un ticket de soporte hardware con fotos del problema.",
    }
    p = problema.lower()
    if "lento" in p or "tarda" in p:
        return soluciones["lento"]
    elif "ruido" in p or "ventilador" in p:
        return soluciones["ruido"]
    return soluciones["default"]


herramientas = [buscar_solucion_software, buscar_solucion_red, buscar_solucion_hardware]
print("Tools:", [h.name for h in herramientas])

## Paso 3 — Nodos

In [ ]:
def clasificar(state: EstadoIT) -> dict:
    """Clasifica el ticket en software, red o hardware."""
    ticket = state["messages"][-1].content
    r = llm.invoke(
        f"Clasifica este ticket IT en UNA sola palabra: 'software', 'red' o 'hardware'.\n"
        f"Si no encaja en ninguna, responde 'desconocido'.\n\nTicket: {ticket}"
    )
    categoria = r.content.strip().lower()
    if categoria not in ["software", "red", "hardware"]:
        categoria = "desconocido"
    print(f"  → categoría: {categoria}")
    return {"categoria": categoria}


def buscar(state: EstadoIT) -> dict:
    """Llama a la herramienta correcta según la categoría."""
    categoria = state["categoria"]
    ticket    = state["messages"][-1].content
    if categoria == "software":
        solucion = buscar_solucion_software.invoke({"problema": ticket})
    elif categoria == "red":
        solucion = buscar_solucion_red.invoke({"problema": ticket})
    elif categoria == "hardware":
        solucion = buscar_solucion_hardware.invoke({"problema": ticket})
    else:
        solucion = "Este ticket será escalado a un técnico. Recibirás respuesta en menos de 24h."
    return {"solucion": solucion}


def responder(state: EstadoIT) -> dict:
    """Redacta una respuesta amable con la solución."""
    ticket   = state["messages"][-1].content
    solucion = state["solucion"]
    prompt = (
        f"Eres un agente de soporte IT amable y profesional. "
        f"El usuario reportó: '{ticket}'.\n\n"
        f"La base de conocimiento sugiere:\n{solucion}\n\n"
        f"Redacta una respuesta breve (máximo 3 frases) dirigida al usuario con los pasos a seguir."
    )
    r = llm.invoke(prompt)
    return {"messages": [AIMessage(content=r.content)]}


def enrutar(state: EstadoIT) -> str:
    """Todos los tickets van a 'buscar' — la herramienta correcta se elige dentro del nodo."""
    return "buscar"


print("Nodos definidos.")

## Paso 4 — Grafo

In [ ]:
g = StateGraph(EstadoIT)

g.add_node("clasificar", clasificar)
g.add_node("buscar",     buscar)
g.add_node("responder",  responder)

g.set_entry_point("clasificar")

# Edge condicional: después de clasificar, enrutar() decide el siguiente nodo
g.add_conditional_edges("clasificar", enrutar, {"buscar": "buscar"})
g.add_edge("buscar",    "responder")
g.add_edge("responder", END)

app = g.compile()
print("Grafo compilado.")

## Paso 5 — Prueba

In [ ]:
tickets = [
    "No puedo instalar el software de contabilidad, me da error de licencia.",
    "No tengo acceso a internet desde esta mañana, el WiFi no conecta.",
    "Mi ordenador hace ruido raro y va muy lento, creo que es el ventilador.",
]

for ticket in tickets:
    print(f"\n{'='*55}")
    print(f"TICKET: {ticket}")
    resultado = app.invoke({
        "messages":  [HumanMessage(content=ticket)],
        "categoria": "",
        "solucion":  "",
    })
    print(f"CATEGORÍA: {resultado['categoria']}")
    print(f"RESPUESTA: {resultado['messages'][-1].content}")

---
## Bonus resuelto — Human-in-the-Loop

El agente propone la solución y espera confirmación antes de enviarla.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

class EstadoITHITL(TypedDict):
    messages:  Annotated[list, add_messages]
    categoria: str
    solucion:  str
    aprobada:  bool

def clasificar_hitl(state: EstadoITHITL) -> dict:
    ticket = state["messages"][-1].content
    r = llm.invoke(f"Clasifica en 'software', 'red' o 'hardware':\n{ticket}")
    cat = r.content.strip().lower()
    if cat not in ["software", "red", "hardware"]:
        cat = "desconocido"
    return {"categoria": cat}

def buscar_hitl(state: EstadoITHITL) -> dict:
    ticket = state["messages"][-1].content
    cat    = state["categoria"]
    if cat == "software":
        sol = buscar_solucion_software.invoke({"problema": ticket})
    elif cat == "red":
        sol = buscar_solucion_red.invoke({"problema": ticket})
    else:
        sol = buscar_solucion_hardware.invoke({"problema": ticket})
    # Propone la solución y espera aprobación
    return {
        "solucion": sol,
        "messages": [AIMessage(content=f"Solución propuesta:\n{sol}\n\n¿La enviamos al usuario?")]
    }

def enviar(state: EstadoITHITL) -> dict:
    return {"messages": [AIMessage(content=f"Respuesta enviada al usuario.\n\n{state['solucion']}")]}

def no_enviar(state: EstadoITHITL) -> dict:
    return {"messages": [AIMessage(content="Solución descartada. El ticket queda pendiente de revisión manual.")]}

def decidir_envio(state: EstadoITHITL) -> str:
    return "enviar" if state.get("aprobada") else "no_enviar"

checkpointer = MemorySaver()
g3 = StateGraph(EstadoITHITL)
g3.add_node("clasificar", clasificar_hitl)
g3.add_node("buscar",     buscar_hitl)
g3.add_node("enviar",     enviar)
g3.add_node("no_enviar",  no_enviar)
g3.set_entry_point("clasificar")
g3.add_edge("clasificar", "buscar")
g3.add_conditional_edges("buscar", decidir_envio, {"enviar": "enviar", "no_enviar": "no_enviar"})
g3.add_edge("enviar",    END)
g3.add_edge("no_enviar", END)

app3 = g3.compile(checkpointer=checkpointer, interrupt_after=["buscar"])
print("Grafo HITL compilado.")

In [ ]:
config = {"configurable": {"thread_id": "ticket-001"}}

# Paso 1: el agente clasifica y propone solución
print("=== PASO 1: el agente propone ===")
estado = app3.invoke({
    "messages":  [HumanMessage(content="No puedo instalar el software de contabilidad, error de licencia.")],
    "categoria": "", "solucion": "", "aprobada": False
}, config=config)
print(estado["messages"][-1].content)
print("\n>>> PAUSADO — esperando aprobación <<<")

# Paso 2: el agente de soporte aprueba
print("\n=== PASO 2: aprobamos ===")
app3.update_state(config, {"aprobada": True})
resultado = app3.invoke(None, config=config)
print(resultado["messages"][-1].content)

---\n# Ejercicio 2 RESUELTO — Agente Text-to-SQL ⭐⭐⭐"

In [ ]:
import sqlite3
import pandas as pd
import re

conn = sqlite3.connect(":memory:")
conn.executescript("""
CREATE TABLE productos (
    id INTEGER PRIMARY KEY, nombre TEXT, categoria TEXT, precio REAL, stock INTEGER
);
CREATE TABLE ventas (
    id INTEGER PRIMARY KEY, producto_id INTEGER, fecha TEXT,
    cantidad INTEGER, canal TEXT,
    FOREIGN KEY (producto_id) REFERENCES productos(id)
);
INSERT INTO productos VALUES
    (1,'MacBook Pro M4','ordenadores',2499.99,45),
    (2,'iPhone 16 Pro','moviles',1199.99,120),
    (3,'iPad Air','tablets',749.99,78),
    (4,'AirPods Pro','audio',279.99,200),
    (5,'Apple Watch','wearables',399.99,95);
INSERT INTO ventas VALUES
    (1,1,'2025-01-10',3,'online'),(2,2,'2025-01-11',8,'tienda'),
    (3,2,'2025-01-12',5,'online'),(4,4,'2025-01-13',15,'online'),
    (5,3,'2025-01-14',4,'tienda'),(6,1,'2025-01-15',2,'online'),
    (7,5,'2025-01-16',6,'tienda'),(8,2,'2025-01-17',10,'online'),
    (9,4,'2025-01-18',20,'tienda'),(10,3,'2025-01-19',3,'online');
""")
conn.commit()

ESQUEMA = """
Tabla productos: id, nombre, categoria, precio (EUR), stock (unidades)
Tabla ventas: id, producto_id (FK->productos.id), fecha (YYYY-MM-DD), cantidad (unidades), canal (online/tienda)
"""

print("Base de datos lista.")
print(pd.read_sql("SELECT * FROM productos", conn))

In [ ]:
class EstadoSQL(TypedDict):
    messages:      Annotated[list, add_messages]
    sql_generado:  str
    resultado_sql: str
    hay_error:     bool


@tool
def ejecutar_sql(query: str) -> str:
    """Ejecuta una consulta SQL contra la base de datos y devuelve el resultado como texto."""
    try:
        df = pd.read_sql(query, conn)
        return df.to_string(index=False)
    except Exception as e:
        return f"ERROR: {e}"


def generar_sql(state: EstadoSQL) -> dict:
    pregunta = state["messages"][-1].content
    prompt = (
        f"Eres experto en SQL con SQLite. Dado el esquema, genera SOLO la consulta SQL "
        f"sin explicaciones ni bloques de código markdown.\n\n"
        f"Esquema:\n{ESQUEMA}\n"
        f"Pregunta: {pregunta}\n\nSQL:"
    )
    r = llm.invoke(prompt)
    sql = re.sub(r"^```(?:sql)?\s*\n?", "", r.content.strip())
    sql = re.sub(r"\n?```\s*$", "", sql).strip()
    print(f"  → SQL: {sql}")
    return {"sql_generado": sql}


def ejecutar(state: EstadoSQL) -> dict:
    resultado = ejecutar_sql.invoke({"query": state["sql_generado"]})
    hay_error = resultado.startswith("ERROR")
    return {"resultado_sql": resultado, "hay_error": hay_error}


def interpretar(state: EstadoSQL) -> dict:
    pregunta  = state["messages"][-1].content
    resultado = state["resultado_sql"]
    prompt = (
        f"El usuario preguntó: '{pregunta}'.\n"
        f"El resultado de la consulta SQL es:\n{resultado}\n\n"
        f"Responde en lenguaje natural, de forma clara y concisa."
    )
    r = llm.invoke(prompt)
    return {"messages": [AIMessage(content=r.content)]}


def responder_error(state: EstadoSQL) -> dict:
    return {"messages": [AIMessage(content=f"No pude ejecutar la consulta. Error: {state['resultado_sql']}")]}


def enrutar_sql(state: EstadoSQL) -> str:
    return "responder_error" if state["hay_error"] else "interpretar"


# Grafo
g_sql = StateGraph(EstadoSQL)
g_sql.add_node("generar_sql",    generar_sql)
g_sql.add_node("ejecutar",       ejecutar)
g_sql.add_node("interpretar",    interpretar)
g_sql.add_node("responder_error", responder_error)

g_sql.set_entry_point("generar_sql")
g_sql.add_edge("generar_sql", "ejecutar")
g_sql.add_conditional_edges("ejecutar", enrutar_sql, {
    "interpretar":    "interpretar",
    "responder_error": "responder_error",
})
g_sql.add_edge("interpretar",    END)
g_sql.add_edge("responder_error", END)

app_sql = g_sql.compile()
print("Grafo Text-to-SQL compilado.")

In [ ]:
preguntas = [
    "¿Cuántas unidades se han vendido en total de cada producto?",
    "¿Cuál es el producto más caro?",
    "¿Cuánto dinero se generó en ventas online vs tienda?",
    "¿Qué producto tiene más stock disponible?",
    "Dame el top 3 de productos por volumen de ventas en euros.",
    "¿Qué día se vendieron más unidades en total?",
]

for pregunta in preguntas:
    print(f"\n{'='*55}")
    print(f"PREGUNTA: {pregunta}")
    resultado = app_sql.invoke({
        "messages":      [HumanMessage(content=pregunta)],
        "sql_generado":  "",
        "resultado_sql": "",
        "hay_error":     False,
    })
    print(f"SQL:      {resultado['sql_generado']}")
    print(f"RESPUESTA: {resultado['messages'][-1].content}")